# Visualizing enVision data

In [11]:
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted

rows = list(string.ascii_uppercase[:16])  # A-P
cols = list(range(1, 25))                 # 1-24

wells = [f"{r}{c:02d}" for r in rows for c in cols]

dataset_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                    "Rhythms/Live Study/enVision/enVision_pre"
                    "processed/2026-05-11._.PREPROCESSED.csv")

raw_data_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                     "Rhythms/Live Study/enVision/enVision_for"
                     "matted/2026-05-11._.FORMATTED.tsv")

platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
)

# dataset_path = Path(
#     "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#     "Study/enVision/enVision_preprocessed/2026-04-01._."
#     "PREPROCESSED.csv"
#     )

# raw_data_path = Path(
#     "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live"
#     " Study/enVision/enVision_formatted/2026-04-01._.FORMATTED.tsv"
#     )

# platemap = (
#     pl.read_excel(
#     "/home/derfelt/PartchLabFiles/Circadian "
#     "Rhythms/Live Study/platemaps/2026-04-01Echo_transfer.xlsx", engine="openpyxl")
#     .select(["384_Well", 
#              "MoleculeID", 
#              "strain",
#              "dex"])
#     .unique(subset=["384_Well"])
#     .with_columns(
#         pl.col("MoleculeID").str.strip_chars()
#     )
#     )

well_excluded = set(wells) - set(platemap["384_Well"].to_list())

print(f"Number of wells not in platemap: {len(well_excluded)}")
# print(*well_excluded, sep="\n")

display(platemap)
# for name, df in platemap.group_by(["MoleculeID", "strain", "dex"]):
#     print(name)
#     display(df)
    # if name[0] == "LY 5uM":
    #     print(name)
    #     display(df)

Number of wells not in platemap: 243


384_Well,MoleculeID,strain,sample_type
str,str,str,str
"""L05""","""0.25% DMSO""","""per2""","""dex"""
"""G21""","""0.0078% DMSO""","""bmal1""","""dex"""
"""F16""","""MK-8692 1.65uM""","""bmal1""","""dex"""
"""H13""","""Dex_Chroma_3TP_bmal1""","""bmal1""",null
"""K04""","""MK-8692 25uM""","""per2""","""dex"""
…,…,…,…
"""G10""","""0.125% DMSO""","""bmal1""","""dex"""
"""J20""","""MK-8800 825nM""","""per2""","""dex"""
"""L03""","""MK-8692 412.5nM""","""per2""","""dex"""


In [2]:
MK = platemap.filter(pl.col("MoleculeID").str.starts_with("MK-"))
MK = MK.with_columns(
    pl.col("MoleculeID").str.split(" ").list.first().alias("MoleculeName")
)["MoleculeName"].unique().sort().to_list()

print(*MK, sep="\n")

MK-8692
MK-8800


In [12]:
def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

In [8]:
raw_timeseries = pl.read_csv(raw_data_path, separator="\t")
raw_timeseries = raw_timeseries.join(platemap, left_on="Well", right_on="384_Well", how="left")

wells = raw_timeseries["Well"].unique().sort().to_list()
# out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/raw_timeseries.pdf")
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/enVision/enVision_visual_results/2026-05-11")

out_path = out_path / "raw_timeseries.pdf"

timepoint_col = "Timepoint"
value = "CalcResultI"
meta_cols = ['MoleculeID', 'strain', 'dex']

with PdfPages(out_path) as pdf:
    g = sns.FacetGrid(
        data=raw_timeseries,
        col="Well",
        col_wrap=24,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, color="red", linewidth=1.5)
    
    # add marks and set ticks after mapping
    x_min = raw_timeseries['Timepoint'].min()
    x_max = raw_timeseries['Timepoint'].max()
    tick_marks = np.arange(np.ceil(x_min / 12) * 12, x_max + 12, 12)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc="Well progress"):
        well_df = (raw_timeseries
              .filter((pl.col("Well") == well))
              .sort("Timepoint", descending=False)
              .to_pandas()
              )
        
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2, markers=False, marker="o", markersize=2)
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        period_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well progress: 100%|██████████| 384/384 [00:48<00:00,  7.90it/s]


## We now need to do the same exact thing but for detrended data

In [14]:
trunc_start = 40
dataset = (
    pl.read_csv(dataset_path)
    .filter(pl.col("feature") == "CalcResultI")
    .select(["Well", "Timepoint", "mean"])
    .filter(pl.col("Timepoint") >= trunc_start) #lets see if truncating shows us the full pic
    .join(platemap, left_on="Well", right_on="384_Well", how="left")
    .sort(['Well', 'Timepoint'])
    )

# smoothing option
dataset = (dataset
           .group_by("Well")
           .map_groups(
               lambda g: 
                   g.with_columns(
                       pl.Series("mean", savgol_filter(
                           g["mean"].to_numpy(),
                           window_length=13,
                           polyorder=3
                       ))
                       )
                   )
           .sort(["Well", "Timepoint"])
)

display(dataset)

wells = dataset["Well"].unique().sort().to_list()
# out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/detrended_savgol_smoothed_timeseries.pdf")

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/enVision/enVision_visual_results/2026-05-11")
out_path = out_path / "detrended_savgol_smoothed_timeseries.pdf"

timepoint_col = "Timepoint"
value = "mean"
meta_cols = ['MoleculeID', 'strain', 'dex']

with PdfPages(out_path) as pdf:
    g = sns.FacetGrid(
        data=dataset,
        col="Well",
        col_wrap=24,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, color="red", linewidth=1.5)
    
    # add marks and set ticks after mapping
    x_min = dataset['Timepoint'].min()
    x_max = dataset['Timepoint'].max()
    tick_marks = np.arange(np.ceil(x_min / 12) * 12, x_max + 12, 12)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc="Well progress"):
        well_df = (dataset
              .filter((pl.col("Well") == well))
              .sort("Timepoint", descending=False)
              .to_pandas()
              )
        
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        # sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2 marker="o", markersize=2)
        sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2)
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        period_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])
        ax.axhline(0, color='gray', linestyle='--', alpha=0.2)
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well,Timepoint,mean,MoleculeID,strain,sample_type
str,f64,f64,str,str,str
"""A01""",40.0,-2.40856,null,null,null
"""A01""",40.5,-2.598793,null,null,null
"""A01""",41.0,-2.732301,null,null,null
"""A01""",41.5,-2.809112,null,null,null
"""A01""",42.0,-2.829255,null,null,null
…,…,…,…,…,…
"""P24""",130.5,7.294099,"""food coloring""",null,null
"""P24""",131.0,9.007971,"""food coloring""",null,null
"""P24""",131.5,7.467273,"""food coloring""",null,null


Well progress: 100%|██████████| 384/384 [00:30<00:00, 12.48it/s]


## Now we do this via grouping the meta data and reporting a mean and sd

In [10]:
dataset = (
    pl.read_csv(dataset_path)
    .filter(pl.col("feature") == "CalcResultI")
    .select(["Well", "Timepoint", "mean"])
    .join(platemap, left_on="Well", right_on="384_Well", how="left")
    .sort(['Well', 'Timepoint'])
    )

# smoothing option
dataset = (dataset
           .group_by("Well")
           .map_groups(
               lambda g: 
                   g.with_columns(
                       pl.Series("mean", savgol_filter(
                           g["mean"].to_numpy(),
                           window_length=13,
                           polyorder=3
                       ))
                       )
                   )
           .sort(["strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, False, False])
        #    .sort(["dex","strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, True, False, False])
)

wells = dataset["Well"].unique().sort().to_list()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/")
# out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#                 "Study/enVision/enVision_visual_results/2026-05-11")

out_path = out_path / "grouped_results.pdf"

timepoint_col = "Timepoint"
value = "mean"
# meta_cols = ['MoleculeID', 'strain', 'dex']
meta_cols = ['MoleculeID', 'strain']

dataset = dataset.to_pandas()
dataset['group_label'] = dataset[meta_cols].apply(
    lambda row: ' | '.join(str(v) for v in row), axis=1
)

groups = dataset['group_label'].unique()
print(*groups, sep="\n")

x_min = dataset[timepoint_col].min()
x_max = dataset[timepoint_col].max()
tick_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)

with PdfPages(out_path) as pdf:
    
    # facet
    
    g = sns.FacetGrid(dataset, col="group_label", col_wrap=10, height=3, aspect=2, sharey=False, sharex=True)
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, errorbar='sd', color='red', linewidth=1.5, 
                    err_kws={'alpha': 0.5, 'facecolor': 'red'})

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    g.set_titles(col_template='{col_name}')
    g.set_axis_labels('Time (hours)', 'Deterended and Smoothed Expression')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    for group in tqdm(groups, desc='plotting groups'):
        gdf = dataset[dataset['group_label'] == group]
        
        fig, ax = plt.subplots(figsize=(12,4))
        sns.lineplot(
            data=gdf,
            x=timepoint_col,
            y=value,
            errorbar='sd',
            color='red',
            linewidth=1.5,
            ax=ax,
            err_kws={'alpha': 0.2, 'facecolor': 'red'}
        )
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_title(group)
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('Deterended and Smoothed Expression')
        
        x_min_g = gdf['Timepoint'].min()
        x_max_g = gdf['Timepoint'].max()
        period_marks = np.arange(np.ceil(x_min_g / 12) * 12, x_max_g + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        
# display(dataset)

nan | nan
food_coloring | nan
noDMSO | Per2
T0901317 5uM | Per2
T0901317 10uM | Per2
SR 5uM | Per2
SR 10uM | Per2
PF 1uM | Per2
Nob 50uM | Per2
Nar 50uM | Per2
MK-8800 50uM | Per2
MK-8800 25uM | Per2
MK-8800 100uM | Per2
MK-8692 50uM | Per2
MK-8692 25uM | Per2
MK-8692 100uM | Per2
MK-2267 25uM | Per2
MK-2267 100uM | Per2
MK-1804 50uM | Per2
MK-1804 25uM | Per2
MK-1804 100uM | Per2
MK-1496 50uM | Per2
MK-1496 25uM | Per2
MK-1496 100uM | Per2
MK-1282 50uM | Per2
MK-1282 25uM | Per2
MK-1282 100uM | Per2
MK-0769 50uM | Per2
MK-0769 25uM | Per2
MK-0769 100uM | Per2
MK-0761 50uM | Per2
MK-0761 25uM | Per2
MK-0761 100uM | Per2
MH-23 50uM | Per2
MH-23 25uM | Per2
MH-23 100uM | Per2
MH-20 50uM | Per2
MH-20 25uM | Per2
MH-20 100uM | Per2
MH-13 25uM | Per2
MH-13 100uM | Per2
MH-11 50uM | Per2
MH-11 25uM | Per2
MH-11 100uM | Per2
LY 5uM | Per2
LY 10uM | Per2
KL001 4uM | Per2
KL001 2uM | Per2
KL001 1uM | Per2
Etoposide 4uM | Per2
Etoposide 2uM | Per2
Etoposide 1uM | Per2
E.coli#36 | Per2
Aswad A002

plotting groups: 100%|██████████| 121/121 [00:12<00:00,  9.92it/s]


## We need to group multiple molecules in 1 plot this time

In [2]:
import matplotlib.colors as mcolors

# PLOT_CONFIGS = [
#     {
#         'title': 'DMSO Controls',
#         'drugs': [],
#         'dmso' : ['0.005% DMSO', '0.01% DMSO', '0.05% DMSO',
#                   '0.1% DMSO', '0.25% DMSO', '0.5% DMSO', '1% DMSO'],
#     },
#     {
#         'title': 'MH-11 & MH-23',
#         'drugs': [('MH-11', 'steelblue'), ('MH-23', 'darkorange')],
#         'dmso' : ['0.05% DMSO'],
#     },
#     {
#         'title': 'MH-13 & MH-20',
#         'drugs': [('MH-13', 'steelblue'), ('MH-20', 'darkorange')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title': 'PF & Etoposide',
#         'drugs': [('PF', 'steelblue'), ('Etoposide', 'darkorange')],
#         'dmso' : ['0.01% DMSO'],
#     },
#     {
#         'title': 'LY, SR, Nobiletin, Naringine, T0901317',
#         'drugs': [
#             ('LY', 'steelblue'), ('SR', 'darkorange'),
#             ('Nob', 'green'), ('Nar', 'purple'), ('T0901317', 'brown')
#         ],
#         'dmso' : ['0.05% DMSO', '0.01% DMSO'],
#     },
#     {
#         'title': 'Bacterial & Aswad',
#         'drugs': [
#             ('E.coli#36', 'steelblue'), ('S.pneumo#6', 'darkorange'),
#             ('Aswad A0005', 'green'), ('Aswad A0010', 'purple'), ('Aswad A0024', 'brown')
#         ],
#         'dmso' : ['0.05% DMSO'],
#     },
#     {
#         'title': 'KL001',
#         'drugs': [('KL001', 'steelblue')],
#         'dmso' : ['0.01% DMSO'],
#     },
#     {
#         'title': 'MK Compounds',
#         'drugs': [
#             ('MK-0761', 'steelblue'), ('MK-0769', 'darkorange'),
#             ('MK-1282', 'green'),     ('MK-1496', 'purple'),
#             ('MK-1804', 'brown'),     ('MK-2267', 'red'),
#             ('MK-8692', 'teal'),      ('MK-8800', 'olive'),
#         ],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-0761',
#         'drugs' : [('MK-0761', 'steelblue')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-1282',
#         'drugs' : [('MK-1282', 'green')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-1804',
#         'drugs' : [('MK-1804', 'brown')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-8692',
#         'drugs' : [('MK-8692', 'teal')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-0769',
#         'drugs' : [('MK-0769', 'darkorange')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-1496',
#         'drugs' : [('MK-1496', 'purple')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-2267',
#         'drugs' : [('MK-2267', 'red')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
#     {
#         'title' : 'MK-8800',
#         'drugs' : [('MK-8800', 'olive')],
#         'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
#     },
# ]

PLOT_CONFIGS = [
    {
        'title': 'DMSO Controls',
        'drugs': [],
        'dmso' : ['0.005% DMSO', '0.01% DMSO', '0.05% DMSO',
                  '0.1% DMSO', '0.25% DMSO', '0.5% DMSO', '1% DMSO'],
    },
    # {
    #     'title': 'MH-11 & MH-23',
    #     'drugs': [('MH-11', 'steelblue'), ('MH-23', 'darkorange')],
    #     'dmso' : ['0.05% DMSO'],
    # },
    # {
    #     'title': 'MH-13 & MH-20',
    #     'drugs': [('MH-13', 'steelblue'), ('MH-20', 'darkorange')],
    #     'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    # },
    {
        'title': 'MH-11',
        'drugs': [('MH-11', 'fuchsia')],
        'dmso': ['1% DMSO']
    },
    {
        'title': 'MH-23',
        'drugs': [('MH-23', 'cadetblue')],
        'dmso': ['1% DMSO']
    },
    {
        'title': 'MH-13',
        'drugs': [('MH-13', 'blueviolet')],
        'dmso': ['1% DMSO']
    },
    {
        'title': 'MH-20',
        'drugs': [('MH-20', 'darkolivegreen')],
        'dmso': ['1% DMSO']
    },
    
]

def get_shades(base_color: str, n: int) -> list:
    """light → dark shades of base_color"""
    rgb = mcolors.to_rgb(base_color)
    return [tuple(1 - (1 - c) * (i + 1) / n for c in rgb) for i in range(n)]

def get_grays(n: int) -> list:
    return [str(round(0.3 + 0.6 * i / max(n - 1, 1), 2)) for i in range(n)]

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def plot_config(ax, df, config, strain=None, colored_dmso = True):
    plot_df = df.copy()
    if strain:
        plot_df = plot_df[plot_df['strain'] == strain]
        
    # plot DMSO controls
    
    dmso_list = config['dmso']
    
    if not config['drugs'] and colored_dmso:
        palette = sns.color_palette("tab10", len(dmso_list))
        for dmso, color in zip(dmso_list, palette):
            sub_df = plot_df[plot_df["MoleculeID"] == dmso]
            if sub_df.empty:
                continue
            sns.lineplot(
                data=sub_df, 
                x=timepoint_col, 
                y=value, 
                errorbar='sd', 
                err_kws={'alpha': 0.2, 'facecolor': color}, 
                color=color,
                ax=ax,
                linewidth=2,
                label=dmso,
                zorder=10
            )
    else:
        grays = get_grays(len(dmso_list))
            
        for drug, base_color in config['drugs']:
            sub_df = plot_df[plot_df['MoleculeID'].str.startswith(drug, na=False)]
            concentrations = natsorted(sub_df['MoleculeID'].unique())
            shades = get_shades(base_color, len(concentrations))
            
            for conc, shade in zip(concentrations, shades):
                sub = sub_df[sub_df['MoleculeID'] == conc]
                if sub.empty:
                    continue
                sns.lineplot(
                    data=sub,
                    x=timepoint_col,
                    y=value,
                    errorbar='sd',
                    color=shade,
                    linewidth=1.5,
                    label=conc,
                    ax=ax,
                    err_kws={'alpha': 0.2, 'facecolor': shade}
                )
        
        for dmso, gray in zip(dmso_list, grays):
            sub_df = plot_df[plot_df["MoleculeID"] == dmso]
            if sub_df.empty:
                continue
            sns.lineplot(
                data=sub_df, 
                x=timepoint_col, 
                y=value, 
                errorbar='sd', 
                err_kws={'alpha': 0.2, 'facecolor': gray}, 
                color=gray,
                ax=ax,
                linewidth=2,
                label=dmso,
                zorder=10
            )
        
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    title = config['title']
    if strain:
        title += f' - {strain}'
    
    ax.set_title(title)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Expression')
    ax.legend(fontsize=10, loc='upper right', ncol=2)

    add_period_marks(ax, plot_df['Timepoint'].min(), plot_df['Timepoint'].max())

In [3]:
dataset = (
    pl.read_csv(dataset_path)
    .filter(pl.col("feature") == "CalcResultI")
    .select(["Well", "Timepoint", "mean"])
    .join(platemap, left_on="Well", right_on="384_Well", how="left")
    .sort(['Well', 'Timepoint'])
    )

# smoothing option
dataset = (dataset
           .group_by("Well")
           .map_groups(
               lambda g: 
                   g.with_columns(
                       pl.Series("mean", savgol_filter(
                           g["mean"].to_numpy(),
                           window_length=13,
                           polyorder=3
                       ))
                       )
                   )
           .sort(["dex","strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, True, False, False])
)

# grab only dex
dataset = dataset.filter(pl.col('dex') == 'dex')
display(dataset)

wells = dataset["Well"].unique().sort().to_list()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/MH._.conc_molecule._.grouped_results.pdf")

timepoint_col = "Timepoint"
value = "mean"
meta_cols = ['MoleculeID', 'strain', 'dex']

dataset = dataset.to_pandas()
dataset['group_label'] = dataset[meta_cols].apply(
    lambda row: ' | '.join(str(v) for v in row), axis=1
)

groups = dataset['group_label'].unique()
# print(*groups, sep="\n")
    
strains = dataset['strain'].dropna().unique()

with PdfPages(out_path) as pdf:
    for config in tqdm(PLOT_CONFIGS, desc="Plot Configs"):
        
        fig, axs = plt.subplots(2, 1, figsize=(24,10), sharey=False)
        for ax, strain in zip(axs, strains):
            plot_config(ax=ax, df=dataset, config=config, strain=strain, colored_dmso=True)
            
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)


Well,Timepoint,mean,MoleculeID,strain,dex
str,f64,f64,str,str,str
"""P01""",24.0,-1002.878725,"""food_coloring""",null,"""dex"""
"""P01""",24.5,-883.431681,"""food_coloring""",null,"""dex"""
"""P01""",25.0,-770.470862,"""food_coloring""",null,"""dex"""
"""P01""",25.5,-663.082941,"""food_coloring""",null,"""dex"""
"""P01""",26.0,-560.354595,"""food_coloring""",null,"""dex"""
…,…,…,…,…,…
"""C13""",92.5,-228.975916,"""0.005% DMSO""","""Bmal1""","""dex"""
"""C13""",93.0,-214.186664,"""0.005% DMSO""","""Bmal1""","""dex"""
"""C13""",93.5,-197.650475,"""0.005% DMSO""","""Bmal1""","""dex"""


Plot Configs: 100%|██████████| 5/5 [00:01<00:00,  3.16it/s]
